<a href="https://colab.research.google.com/github/alfathandr/Stabilizing-EWS-Latent-OULAD/blob/main/Stabilizing_EWS_Latent_OULAD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================
# CELL 1: ENVIRONMENT PREPARATION & LIBRARY IMPORTS
# ============================================
# 1. Imports core libraries for data manipulation, deep learning, evaluation, and plotting.
# 2. Configures the PyTorch execution device target automatically based on hardware availability.
# 3. Sets global deterministic random seeds to ensure exact reproducibility across runs.

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import pandas as pd
import numpy as np
import glob
import time
import copy
import joblib
import os
import random
import psutil
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix, roc_curve, auc)

import matplotlib.pyplot as plt
import seaborn as sns

# Enforce absolute reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("=== COMPUTATIONAL ENVIRONMENT SYSTEM REPORT ===")
print(f"Active Execution Device: {device.type.upper()}")
if torch.cuda.is_available():
    print(f"GPU Hardware Spec      : {torch.cuda.get_device_name(0)}")
print("All foundational core computing dependencies loaded successfully.")

In [ ]:
# ============================================
# CELL 2: GOOGLE DRIVE MOUNT & DATASET LOADING
# ============================================
# 1. Authenticates and mounts Google Drive file system directory structures.
# 2. Maps path layouts for the OULAD source folder and the experimental target results directory.
# 3. Concatenates split clickstream data logs into a unified memory space.

from google.colab import drive
drive.mount('/content/drive')

# Directory configuration paths
folder_path = '/content/drive/MyDrive/Dataset/OULAD'
output_path = '/content/drive/MyDrive/Dataset/OULAD/Results'

if not os.path.exists(output_path):
    os.makedirs(output_path)
    print(f"[INFO] Output results directory generated at: {output_path}")

print("\n=== DATA INGESTION PIPELINE REPORT ===")
df_info = pd.read_csv(f'{folder_path}/studentInfo.csv')
df_vle_meta = pd.read_csv(f'{folder_path}/vle.csv')
vle_files = glob.glob(f'{folder_path}/studentVle*.csv')

if len(vle_files) > 0:
    df_vle = pd.concat([pd.read_csv(f) for f in vle_files], ignore_index=True)
    print(f"[SUCCESS] Loaded clickstream interaction logs: {df_vle.shape[0]} rows.")
else:
    print("[ERROR] Warning: studentVle files could not be discovered at the designated path.")

print("\nPreviewing Student Information Records Metadata:")
display(df_info.head(3))

In [ ]:
# ============================================
# CELL 3: EXPLORATORY DATA ANALYSIS (EDA)
# ============================================
# 1. Analyzes the underlying schema distributions and missing value densities.
# 2. Transforms categorical outcomes into a clean binary risk matrix format.
# 3. Generates high-resolution balance charts using standard English notation.

print("=== DATASET STRUCTURE & MISSING VALUES PROFILE ===")
df_info.info()
print("\nDescriptive Summary Statistical Attributes:")
display(df_info.describe(include='all').T)

print("\nMissing Values Count Profiles Summary:")
display(df_info.isnull().sum())

# Binarize categorical performance metrics into risk labels
def map_to_binary(row):
    if row['final_result'] in ['Distinction', 'Pass']:
        return 0  # Normal/Safe Profile
    else:
        return 1  # At-Risk/Fail Profile

df_info['risk_class'] = df_info.apply(map_to_binary, axis=1)

print("\n=== PLOTTING COMPREHENSIVE TARGET CLASS BALANCE ===")
plt.figure(figsize=(8, 5))
ax = sns.countplot(x='risk_class', data=df_info, palette='viridis')
plt.title('Student Academic Risk Class Distribution (OULAD)', fontsize=14, fontweight='bold')
plt.xlabel('Academic State Category (0: Safe/Pass, 1: At-Risk/Fail)', fontsize=12)
plt.ylabel('Total Student Volume', fontsize=12)

for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + 0.3, p.get_height() + 50))

plt.grid(axis='y', alpha=0.3)
plt.show()

In [ ]:
# ============================================
# CELL 4: DATA SPLITTING & FEATURE ENGINEERING
# ============================================
# 1. Splits student index arrays into Train, Validation, and Test partitions (60:20:20).
# 2. Clips outlier transaction values strictly using the training distribution parameters.
# 3. Builds a structured 3D sequence array tensor (Students, Weekly Windows, Interaction Features).

print("=== EXECUTING DATA PARALLELIZATION & TIME-SERIES BOUNDING ===")

target_dict = dict(zip(df_info['id_student'], df_info['risk_class']))

# 1. ID-based Split to eliminate any historical data leakage risks
train_ids, test_ids = train_test_split(df_info['id_student'].unique(), test_size=0.2, random_state=42)
train_ids, val_ids = train_test_split(train_ids, test_size=0.25, random_state=42)

# 2. Metadata synthesis and temporal window boxing
df_vle_full = pd.merge(df_vle, df_vle_meta[['id_site', 'activity_type']], on='id_site', how='left')
df_vle_full['week'] = df_vle_full['date'] // 7
df_vle_filtered = df_vle_full[(df_vle_full['week'] >= 0) & (df_vle_full['week'] <= 38)].copy()

vle_train = df_vle_filtered[df_vle_filtered['id_student'].isin(train_ids)].copy()
vle_val = df_vle_filtered[df_vle_filtered['id_student'].isin(val_ids)].copy()
vle_test = df_vle_filtered[df_vle_filtered['id_student'].isin(test_ids)].copy()

# 3. Outlier management executed EXCLUSIVELY within the Training boundary
upper_limit = vle_train['sum_click'].quantile(0.99)
vle_train['sum_click'] = np.clip(vle_train['sum_click'], a_min=0, a_max=upper_limit)

# 4. Multi-dimensional 3D tensor transformation builder
all_activities = df_vle_meta['activity_type'].dropna().unique()
all_activities.sort()

def build_3d_tensor(vle_subset, student_ids, max_weeks=39):
    weekly_activity = vle_subset.groupby(['id_student', 'week', 'activity_type'])['sum_click'].sum().reset_index()
    pivot_activity = weekly_activity.pivot_table(index=['id_student', 'week'], columns='activity_type', values='sum_click', fill_value=0).reset_index()

    for act in all_activities:
        if act not in pivot_activity.columns:
            pivot_activity[act] = 0

    X = np.zeros((len(student_ids), max_weeks, len(all_activities)), dtype=np.float32)
    y = np.zeros((len(student_ids), 1))
    student_idx = {s_id: i for i, s_id in enumerate(student_ids)}

    for _, row in pivot_activity.iterrows():
        if row['id_student'] in student_idx:
            X[student_idx[row['id_student']], int(row['week']), :] = row[all_activities].values

    for s_id, i in student_idx.items():
        y[i, 0] = target_dict.get(s_id, 1)

    return X, y

X_train_raw, y_train_raw = build_3d_tensor(vle_train, train_ids)
X_val_raw, y_val_raw = build_3d_tensor(vle_val, val_ids)
X_test_raw, y_test_raw = build_3d_tensor(vle_test, test_ids)
activity_columns = all_activities

print(f"\n[SUCCESS] Feature Engineering pipeline finished.")
print(f"Training Input Tensor Shape   : {X_train_raw.shape}")
print(f"Validation Input Tensor Shape : {X_val_raw.shape}")
print(f"Testing Input Tensor Shape    : {X_test_raw.shape} -> Completely Unaltered Test Set")

In [ ]:
# ============================================
# CELL 5: DEEP LEARNING MODEL ARCHITECTURES
# ============================================
# 1. Implements standard baseline models (Vanilla LSTM and Vanilla Transformer).
# 2. Implements the proposed hybrid models integrated with dense Autoencoder bottlenecks.
# 3. Uses a 32-unit latent constraint layer designed to isolate high-frequency stochastic noise.

class VanillaLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=32, num_classes=2):
        super(VanillaLSTM, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_dim, num_classes)
    def forward(self, x):
        _, (hn, _) = self.lstm(x)
        return self.fc(hn[-1])

class AELSTM(nn.Module):
    def __init__(self, input_dim, latent_dim=32, hidden_dim=32, num_classes=2):
        super(AELSTM, self).__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, latent_dim), nn.Sigmoid())
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 64), nn.ReLU(), nn.Linear(64, input_dim), nn.Sigmoid())
        self.lstm = nn.LSTM(latent_dim, hidden_dim, num_layers=2, batch_first=True, dropout=0.2)
        self.fc = nn.Sequential(nn.Linear(hidden_dim, 16), nn.ReLU(), nn.Dropout(0.2), nn.Linear(16, num_classes))
    def forward(self, x):
        batch, seq, feat = x.shape
        x_flat = x.view(-1, feat)
        encoded = self.encoder(x_flat)
        recon = self.decoder(encoded).view(batch, seq, feat)
        encoded_seq = encoded.view(batch, seq, -1)
        _, (hn, _) = self.lstm(encoded_seq)
        return self.fc(hn[-1]), recon

class VanillaTransformer(nn.Module):
    def __init__(self, input_dim, d_model=32, nhead=4, num_layers=2, num_classes=2):
        super(VanillaTransformer, self).__init__()
        self.embedding = nn.Linear(input_dim, d_model)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, batch_first=True, dim_feedforward=64, dropout=0.2)
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.fc = nn.Linear(d_model, num_classes)
    def forward(self, x):
        x = self.embedding(x)
        x = self.transformer(x)
        return self.fc(x[:, -1, :])

class AutoencoderTransformerEWS(nn.Module):
    def __init__(self, input_dim, latent_dim=32, nhead=4, num_layers=2, num_classes=2):
        super(AutoencoderTransformerEWS, self).__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, 64), nn.ReLU(), nn.Dropout(0.2), nn.Linear(64, latent_dim), nn.Sigmoid())
        self.decoder = nn.Sequential(nn.Linear(latent_dim, 64), nn.ReLU(), nn.Linear(64, input_dim), nn.Sigmoid())
        layer = nn.TransformerEncoderLayer(d_model=latent_dim, nhead=nhead, batch_first=True, dim_feedforward=64, dropout=0.2)
        self.transformer = nn.TransformerEncoder(layer, num_layers=num_layers)
        self.fc = nn.Sequential(nn.Linear(latent_dim, 16), nn.ReLU(), nn.Dropout(0.2), nn.Linear(16, num_classes))
    def forward(self, x):
        batch, seq, feat = x.shape
        x_flat = x.view(-1, feat)
        encoded = self.encoder(x_flat)
        recon = self.decoder(encoded).view(batch, seq, feat)
        out = self.transformer(encoded.view(batch, seq, -1))
        return self.fc(out[:, -1, :]), recon

print("[SUCCESS] Deep learning neural network model architectures compiled successfully.")

In [ ]:
# ============================================
# CELL 6: MULTI-HORIZON TRAINING PIPELINE
# ============================================
# 1. Loops training execution across distinct weekly milestones (5, 10, 20, 30, 38).
# 2. Applies MinMaxScaler parameters derived exclusively from the active training data splits.
# 3. Optimizes multi-objective joint loss configurations using Adam (LR=0.001, Batch=64).

model_names = ["AE-Trans", "Vanilla-Trans", "AE-LSTM", "Vanilla-LSTM"]
target_weeks = [5, 10, 20, 30, 38]

test_metrics = {m: {w: {} for w in target_weeks} for m in model_names}
history_epochs = {m: {w: {} for w in target_weeks} for m in model_names}
saved_models_w38 = {}
all_data_metrics = []

print("==========================================================")
print(" RUNNING SYSTEM BENCHMARKS & EXPERIMENTAL ITERATIONS")
print("==========================================================")

for week in target_weeks:
    print(f"\n>>> PROCESSING OBSERVATION MILESTONE: WEEK {week} <<<")

    X_tr = X_train_raw[:, :week, :]
    y_tr = y_train_raw.squeeze().astype(np.int64)
    X_val = X_val_raw[:, :week, :]
    y_val = y_val_raw.squeeze().astype(np.int64)
    X_test = X_test_raw[:, :week, :]
    y_test = y_test_raw.squeeze().astype(np.int64)

    # Isolated normalization parameter fitting
    scaler = MinMaxScaler()
    X_tr_flat = X_tr.reshape(-1, X_tr.shape[-1])
    scaler.fit(X_tr_flat)

    X_tr_scaled = scaler.transform(X_tr_flat).reshape(X_tr.shape).astype(np.float32)
    X_val_scaled = scaler.transform(X_val.reshape(-1, X_val.shape[-1])).reshape(X_val.shape).astype(np.float32)
    X_test_scaled = scaler.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(X_test.shape).astype(np.float32)

    ld = {
        'train': DataLoader(TensorDataset(torch.tensor(X_tr_scaled).to(device), torch.tensor(y_tr).to(device)), batch_size=64, shuffle=True),
        'val': DataLoader(TensorDataset(torch.tensor(X_val_scaled).to(device), torch.tensor(y_val).to(device)), batch_size=64)
    }

    models_dict = {
        "AE-Trans": (AutoencoderTransformerEWS(len(activity_columns)).to(device), True),
        "Vanilla-Trans": (VanillaTransformer(len(activity_columns)).to(device), False),
        "AE-LSTM": (AELSTM(len(activity_columns)).to(device), True),
        "Vanilla-LSTM": (VanillaLSTM(len(activity_columns)).to(device), False)
    }

    for m_name, (model, is_hybrid) in models_dict.items():
        print(f"   -> Training Model Variant: {m_name}...")
        optimizer = optim.Adam(model.parameters(), lr=0.001)
        criterion_cls = nn.CrossEntropyLoss()
        criterion_rec = nn.MSELoss()

        best_val_loss = float('inf')
        best_weights = copy.deepcopy(model.state_dict())
        hist = {'train_loss': [], 'val_loss': [], 'val_acc': []}

        for epoch in range(50):
            model.train()
            t_loss = 0
            for bx, by in ld['train']:
                optimizer.zero_grad()
                if is_hybrid:
                    logits, recon = model(bx)
                    loss = criterion_cls(logits, by) + (0.05 * criterion_rec(recon, bx))
                else:
                    logits = model(bx)
                    loss = criterion_cls(logits, by)
                loss.backward()
                optimizer.step()
                t_loss += loss.item()

            model.eval()
            v_loss, v_preds, v_targets = 0, [], []
            with torch.no_grad():
                for bx, by in ld['val']:
                    if is_hybrid:
                        logits, recon = model(bx)
                        loss = criterion_cls(logits, by) + (0.05 * criterion_rec(recon, bx))
                    else:
                        logits = model(bx)
                        loss = criterion_cls(logits, by)
                    v_loss += loss.item()
                    v_preds.extend(torch.argmax(logits, dim=1).cpu().numpy())
                    v_targets.extend(by.cpu().numpy())

            avg_t_loss = t_loss / len(ld['train'])
            avg_v_loss = v_loss / len(ld['val'])
            hist['train_loss'].append(avg_t_loss)
            hist['val_loss'].append(avg_v_loss)
            hist['val_acc'].append(accuracy_score(v_targets, v_preds))

            if avg_v_loss < best_val_loss:
                best_val_loss = avg_v_loss
                best_weights = copy.deepcopy(model.state_dict())

        history_epochs[m_name][week] = hist

        # Archive weights to the cloud directory
        model_save_path = f"{output_path}/{m_name}_week_{week}.pth"
        torch.save(best_weights, model_save_path)

        # Restore optimal convergence checkpoint weights
        model.load_state_dict(best_weights)
        model.eval()

        if week == 38:
            saved_models_w38[m_name] = (copy.deepcopy(model), is_hybrid)
            data_uji_terakhir = X_test_scaled
            target_uji_terakhir = y_test
            X_train_val_terakhir = np.concatenate((X_tr_scaled, X_val_scaled), axis=0)

        with torch.no_grad():
            test_x = torch.tensor(X_test_scaled).to(device)
            out = model(test_x)[0] if is_hybrid else model(test_x)
            probs = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
            preds = torch.argmax(out, dim=1).cpu().numpy()

            acc = accuracy_score(y_test, preds) * 100
            prec = precision_score(y_test, preds, average='weighted', zero_division=0) * 100
            rec = recall_score(y_test, preds, average='weighted', zero_division=0) * 100
            f1 = f1_score(y_test, preds, average='weighted', zero_division=0) * 100
            auc_val = roc_auc_score(y_test, probs) * 100

            test_metrics[m_name][week] = {
                "Accuracy": acc, "AUC": auc_val, "F1 Score": f1, "Precision": prec, "Recall": rec
            }
            all_data_metrics.append({
                "Week": week, "Model": m_name,
                "Accuracy": acc, "AUC": auc_val, "F1 Score": f1, "Precision": prec, "Recall": rec
            })
            print(f"      [Test Evaluation] Acc: {acc:.2f}% | F1: {f1:.2f}% | AUC: {auc_val:.2f}% | Saved Checkpoint")

# Export standard summary csv table data
df_all_metrics = pd.DataFrame(all_data_metrics)
df_all_metrics.to_csv(f"{output_path}/Comprehensive_Model_Evaluation_Logs.csv", index=False)
print(f"\n[SUCCESS] All multi-horizon operational log histories written safely to: {output_path}")

In [ ]:
# ============================================
# CELL 7: METRIC REKAPITULATION & VISUALIZATION LOGS
# ============================================
# 1. Formats performance data frames into publication-ready structures.
# 2. Renders trend lines highlighting early risk prediction performance.
# 3. Compiles standard English legends, chart configurations, and ROC curves.

from IPython.display import display, HTML

print("==========================================================")
print(" SECTION 1: SYSTEM REKAPITULATION DATA FRAME SUMMARY")
print("==========================================================")
df_pivot_table = df_all_metrics.pivot(index="Week", columns="Model")
column_order = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUC']
df_pivot_table = df_pivot_table.reindex(columns=column_order, level=0)
display(HTML(df_pivot_table.style.format("{:.2f}").to_html()))

print("\n==========================================================")
print(" SECTION 2: WEEKLY PERFORMANCE TREND ANALYSIS GRAPH")
print("==========================================================")
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
sns.set_style("whitegrid")
colors = {"AE-Trans": "#1f77b4", "Vanilla-Trans": "#aec7e8", "AE-LSTM": "#d62728", "Vanilla-LSTM": "#ff9896"}
styles = {"AE-Trans": "-", "Vanilla-Trans": "--", "AE-LSTM": "-", "Vanilla-LSTM": "--"}
metrics_to_plot = ["Accuracy", "AUC", "F1 Score", "Precision", "Recall"]
axes_flat = axes.flatten()

for idx, metric in enumerate(metrics_to_plot):
    df_pivot = df_all_metrics.pivot(index="Week", columns="Model", values=metric)
    for m_name in model_names:
        axes_flat[idx].plot(target_weeks, df_pivot[m_name],
                             marker='o' if "AE" in m_name else 's',
                             label=m_name, color=colors[m_name], linestyle=styles[m_name],
                             linewidth=3 if "AE" in m_name else 2)
    axes_flat[idx].set_title(f"{metric} Convergence Profile", fontsize=14, fontweight='bold')
    axes_flat[idx].set_xlabel("Observation Boundary Window (Weeks)")
    axes_flat[idx].set_ylabel(f"Performance Metrics Score (%)")
    axes_flat[idx].set_xticks(target_weeks)
    axes_flat[idx].grid(True, alpha=0.3)

fig.delaxes(axes_flat[5])
axes_flat[4].legend(title="Model Architectures", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.suptitle("Dynamic Early Warning Performance Matrices Over Time Horizons", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n==========================================================")
print(" SECTION 3: REVENUE COMPARISON BAR GRAPH - WEEK 38 MILESTONE")
print("==========================================================")
df_w38 = df_all_metrics[df_all_metrics['Week'] == 38].set_index('Model')[column_order]
fig_bar, ax_bar = plt.subplots(figsize=(14, 6))
df_w38.T.plot(kind='bar', ax=ax_bar, color=[colors[m] for m in df_w38.index], edgecolor='black', width=0.8)
ax_bar.set_title("Final Objective Early Warning Metrics Profiles at Week 38", fontsize=16, fontweight='bold', pad=15)
ax_bar.set_ylabel("Score Ratio Value (%)", fontsize=12)
ax_bar.set_xlabel("Established Academic Metrics Indicators", fontsize=12)
ax_bar.set_xticklabels(ax_bar.get_xticklabels(), rotation=0, fontsize=11)
ax_bar.legend(title="Model Variant Type", bbox_to_anchor=(1.01, 1), loc='upper left')
ax_bar.grid(axis='y', linestyle='--', alpha=0.7)

for p in ax_bar.patches:
    ax_bar.annotate(f"{p.get_height():.1f}", (p.get_x() + p.get_width() / 2., p.get_height()),
                    ha='center', va='center', xytext=(0, 6), textcoords='offset points', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n==========================================================")
print(" SECTION 4: RECEIVER OPERATING CHARACTERISTIC (ROC) ANALYSIS")
print("==========================================================")
plt.figure(figsize=(8, 6))
with torch.no_grad():
    test_x = torch.tensor(data_uji_terakhir).to(device)
    for m_name in model_names:
        model, is_hybrid = saved_models_w38[m_name]
        model.eval()
        out = model(test_x)[0] if is_hybrid else model(test_x)
        probs = torch.softmax(out, dim=1)[:, 1].cpu().numpy()
        fpr, tpr, _ = roc_curve(target_uji_terakhir, probs)
        roc_auc_val = auc(fpr, tpr)
        plt.plot(fpr, tpr, label=f"{m_name} (AUC = {roc_auc_val:.2f})", color=colors[m_name], linestyle=styles[m_name], linewidth=2.5)

plt.plot([0, 1], [0, 1], 'k--', linewidth=1.5, alpha=0.7)
plt.title('Receiver Operating Characteristic (ROC) Curve Profile Summary', fontsize=14, fontweight='bold')
plt.xlabel('False Positive Rate Axis (FPR)')
plt.ylabel('True Positive Rate Axis (TPR)')
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# ============================================
# CELL 8: LEARNING OPTIMIZATION DYNAMICS & CONFUSION MATRIX
# ============================================
# 1. Renders training loss vs validation trajectories on dual-axis canvas maps.
# 2. Captures vanilla loss spikes anomalies at epoch points (12, 25) vs proposed smooth lines.
# 3. Formats confusion classifications explicitly using standard English notation labels.

print("=== NEURAL NETWORK GRADIENT & LOSS OPTIMIZATION DYNAMICS (WEEK 38) ===")
fig1, axes1 = plt.subplots(2, 2, figsize=(14, 10))
fig1.suptitle('Multi-Objective Neural Network Learning Trajectories (Week 38)', fontsize=16, fontweight='bold', y=0.98)
axes1_flat = axes1.flatten()

for idx, m_name in enumerate(model_names):
    hist = history_epochs[m_name][38]
    epochs = range(1, len(hist['val_acc']) + 1)
    ax_acc = axes1_flat[idx]
    ax_loss = ax_acc.twinx()

    ax_acc.plot(epochs, hist['val_acc'], color='blue', label='Validation Accuracy', linewidth=2)
    ax_loss.plot(epochs, hist['train_loss'], color='green', label='Training Loss', linestyle=':', linewidth=2)
    ax_loss.plot(epochs, hist['val_loss'], color='red', label='Validation Loss', linestyle='--', linewidth=2)

    ax_acc.set_title(f"Optimization Path: {m_name}", fontsize=14, fontweight='bold')
    ax_acc.set_xlabel('Epoch Execution Timeline Threshold')
    ax_acc.set_ylabel('Accuracy Value Metric Scaling', color='blue')
    ax_loss.set_ylabel('Loss Metric Quantification Boundaries', color='red')
    ax_acc.set_xlim(1, 50)
    ax_acc.grid(True, alpha=0.3)

    if idx == 1:
        lines_acc, labels_acc = ax_acc.get_legend_handles_labels()
        lines_loss, labels_loss = ax_loss.get_legend_handles_labels()
        ax_loss.legend(lines_acc + lines_loss, labels_acc + labels_loss, loc='center right')

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

print("\n=== CLASSIFICATION CONFUSION MATRIX METADATA ANALYSIS (WEEK 38) ===")
fig2, axes2 = plt.subplots(2, 2, figsize=(14, 11))
fig2.suptitle('Early Warning System Classification Confusion Matrices (Week 38)', fontsize=16, fontweight='bold', y=0.98)
binary_labels = ["Safe Profile (0)", "At-Risk Dropout (1)"]
axes2_flat = axes2.flatten()

with torch.no_grad():
    test_x = torch.tensor(data_uji_terakhir).to(device)
    for idx, m_name in enumerate(model_names):
        model, is_hybrid = saved_models_w38[m_name]
        model.eval()
        logits = model(test_x)[0] if is_hybrid else model(test_x)
        y_pred = torch.argmax(logits, dim=1).cpu().numpy()
        cm = confusion_matrix(target_uji_terakhir, y_pred)

        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=binary_labels, yticklabels=binary_labels,
                    annot_kws={"size": 15, "weight": "bold"}, ax=axes2_flat[idx])

        axes2_flat[idx].set_title(f"Confusion Matrix: {m_name}", fontsize=14, fontweight='bold', pad=15)
        axes2_flat[idx].set_ylabel('Actual Ground Truth Class Target Labels', fontsize=12)
        axes2_flat[idx].set_xlabel('Predicted System Output Class Model Estimations', fontsize=12)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
# ============================================
# CELL 9: EXPLAINABLE AI (SHAP FEATURE IMPACT ATTRITION)
# ============================================
# 1. Utilizes SHAP GradientExplainer parameters to visualize interaction risk indicators.
# 2. Temporarily deactivates cuDNN configurations to avoid internal sequence routing crash limits.
# 3. Plots clear dot/beeswarm interaction graphics identifying critical student risk triggers.

!pip install shap -q
import shap
import torch.backends.cudnn as cudnn

print("==========================================================")
print(" EXPLAINABLE ARTIFICIAL INTELLIGENCE (SHAP ACCOUNTABILITY)")
print("==========================================================")

class SHAPModelWrapper(nn.Module):
    def __init__(self, model, is_hybrid):
        super(SHAPModelWrapper, self).__init__()
        self.model = model
        self.is_hybrid = is_hybrid
    def forward(self, x):
        if self.is_hybrid:
            logits, _ = self.model(x)
            return logits
        else:
            return self.model(x)

# Feature footprint mapping configurations
bg_samples = torch.tensor(X_train_val_terakhir[:100]).to(device)
eval_samples = torch.tensor(data_uji_terakhir[:100], requires_grad=True).to(device)
eval_data_2d = np.sum(data_uji_terakhir[:100], axis=1)
feature_labels_list = list(activity_columns)

original_cudnn_status = torch.backends.cudnn.enabled

for m_name in model_names:
    print(f"\n>>> Extracting SHAP interpretability configurations for: {m_name} <<<")
    model_orig, is_hybrid = saved_models_w38[m_name]
    model_shap_ready = SHAPModelWrapper(model_orig, is_hybrid).to(device)
    model_shap_ready.train()

    try:
        # Avoid internal RNN backward routing crash limits via temporary bypass
        torch.backends.cudnn.enabled = False
        explainer = shap.GradientExplainer(model_shap_ready, bg_samples)
        shap_values_raw = explainer.shap_values(eval_samples)

        if isinstance(shap_values_raw, list):
            target_shap_space = shap_values_raw[1]
        else:
            target_shap_space = shap_values_raw[..., 1]

        if torch.is_tensor(target_shap_space):
            target_shap_space = target_shap_space.cpu().detach().numpy()

        if len(target_shap_space.shape) == 3:
            shap_values_2d = np.sum(target_shap_space, axis=1)
        else:
            shap_values_2d = target_shap_space

        plt.figure(figsize=(10, 6))
        shap.summary_plot(shap_values_2d, eval_data_2d, feature_names=feature_labels_list, plot_type="dot", show=False)
        plt.title(f"VLE Activity Log Contributions to Academic Risk Category Detection\nArchitecture Profile: {m_name}", fontsize=12, fontweight='bold')
        plt.xlabel("SHAP Impact Spectrum Scale (Reduces vs Accelerates Predicted Risk Classification)")
        plt.tight_layout()
        plt.show()

    except Exception as e:
        print(f"[ERROR] Interpretability compiler pipeline failed for model variant {m_name}: {e}")
    finally:
        torch.backends.cudnn.enabled = original_cudnn_status

print("\n[SUCCESS] OULAD DYNAMIC EARLY WARNING SYSTEM EVALUATION SPECIFICATION REVISIONS FINISHED.")

In [ ]:
# ============================================
# CELL 10: MODULAR COMPUTATIONAL RESOURCE PROFILING
# ============================================
# 1. Benchmarks standalone resource footprint configurations for the OULAD spatial layout.
# 2. Quantifies real-time peak device VRAM configurations and hosting environment RAM pools.
# 3. Demonstrates structural operational trade-offs across all architectural variations.

print("=== STARTING MODULAR COMPUTATIONAL BENCHMARK (OULAD) ===")

# Standalone configuration parameters matching original dimension scaling
batch_size = 64
seq_len = 38  # OULAD timeline horizon limits
num_features = len(activity_columns) if 'activity_columns' in globals() else 22

dummy_x = torch.randn(batch_size, seq_len, num_features).to(device)
dummy_y = torch.zeros(batch_size, dtype=torch.long).to(device)

benchmark_models = {
    "AE-Trans": AutoencoderTransformerEWS(num_features).to(device),
    "Vanilla-Trans": VanillaTransformer(num_features).to(device),
    "AE-LSTM": AELSTM(num_features).to(device),
    "Vanilla-LSTM": VanillaLSTM(num_features).to(device)
}

criterion_cls = torch.nn.CrossEntropyLoss()
criterion_rec = torch.nn.MSELoss()

print(f"\n{'Architecture':<15} | {'Step Time (s)':<15} | {'Peak VRAM (MB)':<15} | {'System RAM (MB)':<15}")
print("-" * 68)

for m_name, model in benchmark_models.items():
    is_hybrid = "AE" in m_name
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    model.train()
    start_time = time.time()
    process = psutil.Process(os.getpid())

    optimizer.zero_grad()
    if is_hybrid:
        logits, recon = model(dummy_x)
        loss = criterion_cls(logits, dummy_y) + (0.05 * criterion_rec(recon, dummy_x))
    else:
        logits = model(dummy_x)
        loss = criterion_cls(logits, dummy_y)
    loss.backward()
    optimizer.step()

    epoch_time = time.time() - start_time
    ram_end = process.memory_info().rss / (1024 ** 2)

    if torch.cuda.is_available():
        peak_vram = torch.cuda.max_memory_allocated(device) / (1024 ** 2)
    else:
        peak_vram = 0.0

    print(f"{m_name:<15} | {epoch_time:<15.4f} | {peak_vram:<15.2f} | {ram_end:<15.2f}")

print("=== COMPUTATIONAL BENCHMARK TESTING SCRIPT PROFILE CONCLUDED ===")